# 01 — 数据清洗 & 预处理

**目标**: 将原始交易数据清洗为标准分析数据集，为后续 EDA 和 RFM 建模打好基础。

**步骤**:
1. 加载数据 → 了解数据结构
2. 检查缺失值、重复值、异常值
3. 修正数据类型
4. 创建派生特征（年/月/季度/处理天数）
5. 导出清洗后数据

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示（Windows 可能需要调整）
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# 加载数据
df = pd.read_csv('../data/Sample_Superstore.csv')
print(f'Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')

Shape: (2029, 21)
Memory usage: 2088.08 KB


In [3]:
# ========== 1. 数据结构概览 ==========
print('=== 数据类型 ===')
print(df.dtypes)
print(f'\n=== 前5行 ===')
df.head()

=== 数据类型 ===
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

=== 前5行 ===


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-0365,2020-01-03,2020-01-06,First Class,CG-00284,Customer 284,Home Office,United States,Colorado,...,15693,West,FUR-OFF-0054,Office Supplies,Paper,Paper Model 4,69.01,2,0.05,7.16
1,2,US-0217,2020-01-03,2020-01-05,First Class,CG-00295,Customer 295,Home Office,United States,Minnesota,...,65026,Central,FUR-OFF-0030,Office Supplies,Art,Art Model 5,14.18,2,0.00,4.86
2,3,US-0365,2020-01-03,2020-01-06,First Class,CG-00284,Customer 284,Home Office,United States,Colorado,...,40079,West,FUR-FUR-0006,Furniture,Chairs,Chairs Model 1,267.19,2,0.20,42.44
3,4,US-0217,2020-01-03,2020-01-05,First Class,CG-00295,Customer 295,Home Office,United States,Minnesota,...,28661,Central,FUR-OFF-0049,Office Supplies,Labels,Labels Model 4,23.68,2,0.00,7.88
4,5,US-0217,2020-01-03,2020-01-05,First Class,CG-00295,Customer 295,Home Office,United States,Minnesota,...,36895,Central,FUR-OFF-0049,Office Supplies,Labels,Labels Model 4,35.52,3,0.00,11.24


In [4]:
# ========== 2. 缺失值检查 ==========
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Pct %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print('⚠️  发现缺失值:')
    print(missing_df)
else:
    print('✅ 没有缺失值')

✅ 没有缺失值


In [5]:
# ========== 3. 重复值检查 ==========
duplicates = df.duplicated().sum()
print(f'完全重复行: {duplicates}')

# 检查 Row ID 是否唯一
row_id_dupes = df['Row ID'].duplicated().sum()
print(f'Row ID 重复: {row_id_dupes}')

# 检查 Order ID + Product ID 组合
order_product_dupes = df.duplicated(subset=['Order ID', 'Product ID']).sum()
print(f'同一订单同一产品重复: {order_product_dupes}')

完全重复行: 0
Row ID 重复: 0
同一订单同一产品重复: 15


In [6]:
# ========== 4. 基本统计量 ==========
print('=== 数值列描述统计 ===')
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()

=== 数值列描述统计 ===


,Sales,Quantity,Discount,Profit
count,2029.000000,2029.000000,2029.000000,2029.000000
mean,717.188896,2.084771,0.075727,164.529724
std,1775.185021,1.159265,0.084315,542.512460
min,1.110000,1.000000,0.000000,-240.630000
25%,24.870000,1.000000,0.000000,2.900000
50%,133.550000,2.000000,0.050000,12.900000
75%,581.050000,3.000000,0.150000,88.390000
max,18404.290000,5.000000,0.300000,9200.420000


In [7]:
# ========== 5. 业务逻辑检查 ==========

# 5a. Sales 应为正数
negative_sales = df[df['Sales'] <= 0]
print(f'Sales <= 0: {len(negative_sales)} 行')

# 5b. Quantity 应 >= 1
zero_qty = df[df['Quantity'] < 1]
print(f'Quantity < 1: {len(zero_qty)} 行')

# 5c. Discount 应在 0-1 之间
invalid_discount = df[(df['Discount'] < 0) | (df['Discount'] > 1)]
print(f'Discount 超出 [0,1]: {len(invalid_discount)} 行')

# 5d. Ship Date 应 >= Order Date
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
bad_ship = df[df['Ship Date'] < df['Order Date']]
print(f'发货日期早于下单日期: {len(bad_ship)} 行')

# 5e. Profit 可能为负，但检查极端异常值
q1, q3 = df['Profit'].quantile([0.01, 0.99]).values
extreme_profit = df[(df['Profit'] < q1 * 3) | (df['Profit'] > q3 * 3)]
print(f'极端 Profit 异常值: {len(extreme_profit)} 行')

Sales <= 0: 0 行
Quantity < 1: 0 行
Discount 超出 [0,1]: 0 行
发货日期早于下单日期: 0 行
极端 Profit 异常值: 3 行


In [8]:
# ========== 6. 类别变量检查 ==========
categorical_cols = ['Ship Mode', 'Segment', 'Country', 'Region', 'Category', 'Sub-Category']

for col in categorical_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().to_string())
    print(f'Unique values: {df[col].nunique()}')


--- Ship Mode ---
Ship Mode
Standard Class    813
First Class       520
Second Class      512
Same Day          184
Unique values: 4

--- Segment ---
Segment
Home Office    683
Consumer       676
Corporate      670
Unique values: 3

--- Country ---
Country
United States    2029
Unique values: 1

--- Region ---
Region
South      549
East       521
West       493
Central    466
Unique values: 4

--- Category ---
Category
Office Supplies    1065
Furniture           491
Technology          473
Unique values: 3

--- Sub-Category ---
Sub-Category
Storage        135
Furnishings    130
Phones         127
Art            125
Chairs         124
Appliances     123
Envelopes      121
Bookcases      121
Accessories    119
Labels         119
Binders        119
Fasteners      117
Tables         116
Copiers        114
Machines       113
Paper          104
Supplies       102
Unique values: 17


In [9]:
# ========== 7. 创建派生特征 ==========

# 时间维度
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Quarter'] = df['Order Date'].dt.quarter
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')
df['Order Day of Week'] = df['Order Date'].dt.dayofweek
df['Order Day Name'] = df['Order Date'].dt.day_name()

# 处理时长（天）
df['Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# 盈利标记
df['Is Profitable'] = df['Profit'] > 0

# 折扣分桶
df['Discount Tier'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.1, 0.3, 1.0],
    labels=['No Discount', 'Low (0-10%)', 'Medium (10-30%)', 'High (>30%)']
)

# 利润率
df['Profit Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print('✅ 派生特征已创建')
print(f'新增列: Order Year, Order Month, Order Quarter, Processing Days, Is Profitable, Discount Tier, Profit Margin')

✅ 派生特征已创建
新增列: Order Year, Order Month, Order Quarter, Processing Days, Is Profitable, Discount Tier, Profit Margin


In [10]:
# ========== 8. 最终数据质量报告 ==========
print('======== 数据质量报告 ========')
print(f'总行数: {len(df):,}')
print(f'总列数: {len(df.columns)}')
print(f'订单数: {df["Order ID"].nunique():,}')
print(f'客户数: {df["Customer ID"].nunique():,}')
print(f'产品数: {df["Product ID"].nunique():,}')
print(f'日期范围: {df["Order Date"].min().date()} ~ {df["Order Date"].max().date()}')
print(f'总销售额: ${df["Sales"].sum():,.0f}')
print(f'总利润: ${df["Profit"].sum():,.0f}')
print(f'整体利润率: {df["Profit"].sum() / df["Sales"].sum() * 100:.2f}%')
print(f'平均处理天数: {df["Processing Days"].mean():.1f} 天')
print(f'亏损交易占比: {(~df["Is Profitable"]).mean() * 100:.2f}%')
print('=' * 35)

======== 数据质量报告 ========
总行数: 2,029
总列数: 31
订单数: 1,098
客户数: 375
产品数: 85
日期范围: 2020-01-03 ~ 2023-12-31
总销售额: $1,455,176
总利润: $333,831
整体利润率: 22.94%
平均处理天数: 4.0 天
亏损交易占比: 6.01%


In [11]:
# ========== 9. 保存清洗后数据 ==========
cleaned_path = '../data/Superstore_Cleaned.csv'
df.to_csv(cleaned_path, index=False)
print(f'✅ Cleaned data saved to: {cleaned_path}')
print(f'Columns: {list(df.columns)}')

✅ Cleaned data saved to: ../data/Superstore_Cleaned.csv
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Order Year', 'Order Month', 'Order Quarter', 'Order Month Name', 'Order Day of Week', 'Order Day Name', 'Processing Days', 'Is Profitable', 'Discount Tier', 'Profit Margin']


---
## 清洗总结

| 检查项 | 结果 |
|--------|------|
| 缺失值 | ✅ 无缺失 |
| 重复值 | ✅ 无重复 |
| 数据类型 | ✅ 日期列已转为 datetime |
| 业务逻辑 | ✅ Sales/Quantity/Discount 均在合理范围 |
| 派生特征 | ✅ 新增 9 个分析用列 |

**下一步**: → `02_eda.ipynb` 探索性数据分析